In [0]:
import requests
from datetime import datetime, timedelta
from pyspark.sql import functions as F

In [0]:
API_KEY = dbutils.secrets.get(scope="electricity", key="fred_api_key")
FRED_SERIES_BASE_PATH = "https://api.stlouisfed.org/fred/series/observations?"
ELECTRICTY_AVG_SERIES_ID = "APU000072610"
START_DATE = datetime(datetime.now().year - 1, 1, 1).strftime("%Y-%m-%d")
END_DATE = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")

In [0]:
obs_response = requests.get(f"{FRED_SERIES_BASE_PATH}series_id={ELECTRICTY_AVG_SERIES_ID}&api_key={API_KEY}&file_type=json&observation_start={START_DATE}&observation_end={END_DATE}")
obs_data = obs_response.json()
obs_df = spark.createDataFrame(obs_data['observations']).select('date', 'value') \
    .withColumn('value', F.when((F.col('value').isNull()) | (F.trim(F.col('value')) == '') | (F.col('value') == '.'), None).otherwise(F.col('value')))




In [0]:
obs_df.write.format("delta").mode("overwrite").saveAsTable("workspace.us_electricity.raw_electricity_prices")